In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append(os.path.abspath(".."))
os.chdir(os.path.abspath(".."))

with open("keys/openai_api_key.txt") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

with open("keys/hf_token.txt") as f:
    os.environ["HF_TOKEN"] = f.read().strip()
    os.environ["HF_HOME"] = "/network/scratch/t/tejas.kasetty/huggingface"

### Data and Prompt Generation

**Test**

In [14]:
from src.dataset import Dataset, ShiftCipher

sc = ShiftCipher(10, 10)
data = sc.sample(1)

In [15]:
from src.generate import ShiftCipherGenerator

sc_gen = ShiftCipherGenerator()
sc_gen.generate(data)

('You are a codebreaker. Your task is to encode the given text using a shift cipher.\n\n',
 [                                              context  \
  0      decoded: Hippocratic\nencoded: Abiihvktmbv\n\n   
  1            decoded: aviatrix\nencoded: tobtmkbq\n\n   
  2            decoded: airwoman\nencoded: tbkphftg\n\n   
  3          decoded: aviatress\nencoded: tobtmkxll\n\n   
  4                      decoded: nip\nencoded: gbi\n\n   
  5              decoded: puff_up\nencoded: inyy_ni\n\n   
  6          decoded: noble_gas\nencoded: ghuex_ztl\n\n   
  7          decoded: inert_gas\nencoded: bgxkm_ztl\n\n   
  8              decoded: argonon\nencoded: tkzhghg\n\n   
  9                decoded: thawed\nencoded: matpxw\n\n   
  10       decoded: Orthoptera\nencoded: Hkmahimxkt\n\n   
  11  decoded: order_Orthoptera\nencoded: hkwxk_Hkma...   
  12  decoded: hymenopterous\nencoded: arfxghimxkhnl...   
  13           decoded: Guinness\nencoded: Znbggxll\n\n   
  14             decoded

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/mila/t/tejas.kasetty/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


ValueError: a must be 1-dimensional or an integer

In [30]:
models = [ "gpt-3.5-turbo", "gpt-4" ]
llama_models = ["meta-llama/Llama-2-7b-chat-hf", "meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-v0.1", "mistralai/Mistral-7B-Instruct-v0.1", "mistralai/Mistral-7B-Instruct-v0.2", "mistralai/Mistral-7B-v0.3", "mistralai/Ministral-8B-Instruct-2410"]

In [ ]:
from vllm import LLM, SamplingParams
model_name = llama_models[0]
llm = LLM(model=model_name, tensor_parallel_size=2, gpu_memory_utilization=.90) 

/home/mila/t/tejas.kasetty/.conda/envs/preq_code/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 04-25 17:34:05 [__init__.py:239] Automatically detected platform cuda.


2025-04-25 17:34:09,071	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 04-25 17:34:25 [config.py:689] This model supports multiple tasks: {'classify', 'reward', 'score', 'generate', 'embed'}. Defaulting to 'generate'.
WARNING 04-25 17:34:26 [arg_utils.py:1731] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 04-25 17:34:26 [config.py:1713] Defaulting to use ray for distributed inference
INFO 04-25 17:34:26 [llm_engine.py:243] Initializing a V0 LLM engine (v0.8.4) with config: model='meta-llama/Llama-2-7b-chat-hf', speculative_config=None, tokenizer='meta-llama/Llama-2-7b-chat-hf', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_

2025-04-25 17:34:30,381	INFO worker.py:1841 -- Started a local Ray instance.


INFO 04-25 17:34:34 [ray_utils.py:335] No current placement group found. Creating a new placement group.
WARNING 04-25 17:34:34 [ray_utils.py:342] The number of required GPUs exceeds the total number of available GPUs in the placement group.
INFO 04-25 17:34:44 [ray_utils.py:233] Waiting for creating a placement group of specs for 10 seconds. specs=[{'GPU': 1.0, 'node:172.16.11.137': 0.001}, {'GPU': 1.0}]. Check `ray status` and `ray list nodes` to see if you have enough resources, and make sure the IP addresses used by ray cluster are the same as VLLM_HOST_IP environment variable specified in each node if you are running on a multi-node.
INFO 04-25 17:35:04 [ray_utils.py:233] Waiting for creating a placement group of specs for 30 seconds. specs=[{'GPU': 1.0, 'node:172.16.11.137': 0.001}, {'GPU': 1.0}]. Check `ray status` and `ray list nodes` to see if you have enough resources, and make sure the IP addresses used by ray cluster are the same as VLLM_HOST_IP environment variable specifi

In [7]:
 # You can use any Hugging Face causal model
from math import exp
X = list(range(30))
Y = [2 * x + 1 for x in X]

# Create prompt list for different timesteps
prompts = []
for t in range(1, len(X)):
    context = ""
    for k in range(t):
        context += f"x={X[k]}, y={Y[k]}\n"
    context += f"x={X[t]}, y="
    prompts.append(context)

# Predict next token in batch
sampling_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=6)
outputs = llm.generate(prompts, sampling_params)
res = []
# Print predictions
for i, output in enumerate(outputs):
    prediction = output.outputs[0].text.strip()
    out = output
    logprobs = [ (logprob.decoded_token, round(exp(logprob.logprob), 4)) for (token, logprob) in output.outputs[0].logprobs[0].items() ]
    res.append(logprobs[0][1])
    print(logprobs)
    print(f"Prompt {i+1}:\n{prompts[i]}\nPredicted y: {prediction} | Ground truth y: {Y[i+1]}\n{'-'*40}")
print(model_name, res)

Processed prompts: 100%|██████████| 29/29 [00:00<00:00, 246.29it/s, est. speed input: 35636.06 toks/s, output: 246.45 toks/s]

[('0', 0.558), ('2', 0.1702), ('1', 0.1411), ('x', 0.0458), ('3', 0.038), ('5', 0.014)]
Prompt 1:
x=0, y=1
x=1, y=
Predicted y: 0 | Ground truth y: 3
----------------------------------------
[('5', 0.429), ('4', 0.1085), ('6', 0.1019), ('2', 0.0957), ('7', 0.0745), ('1', 0.0618)]
Prompt 2:
x=0, y=1
x=1, y=3
x=2, y=
Predicted y: 5 | Ground truth y: 5
----------------------------------------
[('7', 0.7056), ('6', 0.0544), ('2', 0.048), ('1', 0.048), ('4', 0.0351), ('8', 0.031)]
Prompt 3:
x=0, y=1
x=1, y=3
x=2, y=5
x=3, y=
Predicted y: 7 | Ground truth y: 7
----------------------------------------
[('9', 0.941), ('1', 0.0343), ('8', 0.0118), ('2', 0.0044), ('0', 0.0019), ('3', 0.0017)]
Prompt 4:
x=0, y=1
x=1, y=3
x=2, y=5
x=3, y=7
x=4, y=
Predicted y: 9 | Ground truth y: 9
----------------------------------------
[('1', 0.9819), ('2', 0.0043), ('9', 0.002), ('0', 0.002), ('3', 0.0019), ('8', 0.0011)]
Prompt 5:
x=0, y=1
x=1, y=3
x=2, y=5
x=3, y=7
x=4, y=9
x=5, y=
Predicted y: 1 | Ground tr

In [12]:
for (t, v) in out.outputs[0].logprobs[0].items():
    print(t, v)

2946 Logprob(logprob=-6.627816765103489e-05, rank=1, decoded_token='59')
2970 Logprob(logprob=-11.125065803527832, rank=2, decoded_token='58')
220 Logprob(logprob=-11.937565803527832, rank=3, decoded_token='Ġ')
16 Logprob(logprob=-12.375065803527832, rank=4, decoded_token='1')
24 Logprob(logprob=-12.937565803527832, rank=5, decoded_token='9')
1399 Logprob(logprob=-12.937565803527832, rank=6, decoded_token='60')


In [ ]:

["meta-llama/Llama-3.1-8B-Instruct", [0.6571, 0.8955, 0.9482, 0.9892, 0.9654, 0.992, 0.9986, 0.9972, 0.9993, 0.9906, 0.9989, 0.9991, 0.9992, 0.9993, 0.9994, 0.9992, 0.9998, 0.9997, 0.9999, 0.9997, 0.9998, 0.9999, 0.9999, 0.9999, 0.9998, 0.9997, 0.9997, 0.9999, 0.9999]]
["mistralai/Mistral-7B-v0.1", [0.4459, 0.4893, 0.4478, 0.9, 0.9617, 0.9874, 0.9925, 0.9945, 0.9924, 0.987, 0.9917, 0.9946, 0.9956, 0.9961, 0.9923, 0.9939, 0.9974, 0.9983, 0.9977, 0.9933, 0.9961, 0.9984, 0.9989, 0.9984, 0.997, 0.9981, 0.9991, 0.9992, 0.9985]]
["mistralai/Mistral-7B-v0.3", [0.3512, 0.3977, 0.4113, 0.852, 0.9581, 0.9862, 0.9927, 0.9943, 0.9937, 0.9867, 0.9923, 0.9947, 0.9956, 0.9959, 0.9935, 0.9952, 0.9986, 0.9985, 0.9982, 0.9954, 0.9974, 0.9988, 0.999, 0.9986, 0.9981, 0.9985, 0.9991, 0.9992, 0.999]]
["mistralai/Mistral-7B-Instruct-v0.1", [0.558, 0.429, 0.7056, 0.941, 0.9819, 0.9969, 0.9979, 0.9968, 0.9803, 0.9958, 0.9976, 0.9979, 0.9981, 0.9876, 0.9982, 0.9985, 0.9988, 0.9984, 0.9968, 0.9982, 0.9996, 0.9997, 0.9997, 0.9983, 0.9996, 0.9996, 0.9997, 0.9997, 0.9986]]
["mistralai/Mistral-7B-Instruct-v0.2", [0.397, 0.5195, 0.922, 0.9728, 0.9746, 0.9915, 0.9981, 0.997, 0.9951, 0.9955, 0.9961, 0.9975, 0.999, 0.9981, 0.999, 0.9978, 0.9995, 0.9996, 0.9993, 0.9992, 0.9993, 0.9997, 0.9999, 0.9997, 0.9999, 0.9999, 0.9999, 0.9999, 0.9997]]
["mistralai/Ministral-8B-Instruct-2410", [0.3279, 0.4283, 0.9297, 0.9655, 0.9909, 0.9957, 0.998, 0.9986, 0.9981, 0.9945, 0.9961, 0.9981, 0.9983, 0.9989, 0.9988, 0.9989, 0.9996, 0.9997, 0.9995, 0.9981, 0.9987, 0.9996, 0.9997, 0.9995, 0.9991, 0.9996, 0.9999, 0.9999, 0.9998]]

['mistralai/Mistral-7B-v0.3',
 [0.3512,
  0.3977,
  0.4113,
  0.852,
  0.9581,
  0.9862,
  0.9927,
  0.9943,
  0.9937,
  0.9867,
  0.9923,
  0.9947,
  0.9956,
  0.9959,
  0.9935,
  0.9952,
  0.9986,
  0.9985,
  0.9982,
  0.9954,
  0.9974,
  0.9988,
  0.999,
  0.9986,
  0.9981,
  0.9985,
  0.9991,
  0.9992,
  0.999]]

## Sampling words

In [1]:
import nltk
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/mila/t/tejas.kasetty/nltk_data...


True

In [10]:
def shift_letters(text, shift=0, start=0, step=1):
    def shift_char(c, shift):
        if 'A' <= c <= 'Z':
            return chr((ord(c) - ord('A') + shift) % 26 + ord('A'))
        elif 'a' <= c <= 'z':
            return chr((ord(c) - ord('a') + shift) % 26 + ord('a'))
        return c  # Non-letter characters unchanged

    result = []
    for idx, c in enumerate(text):
        if idx >= start and (idx - start) % step == 0 and c.isalpha():
            c = shift_char(c, shift)
        result.append(c)
    return ''.join(result)


In [13]:
from nltk.corpus import wordnet as wn
import random

# Get all synsets (concepts/meanings)
all_synsets = list(wn.all_synsets())

# Sample N random synsets
sampled_synsets = random.sample(all_synsets, 200)
print(sampled_synsets[2].lemmas()[0].name())
# Extract word lemmas from those synsets
sampled_words = []
for synset in sampled_synsets:
    for lemma in synset.lemmas():
        sampled_words.append(lemma.name())

# Optionally remove duplicates
unique_words = list(set(sampled_words))
shift_cipher = [ shift_letters(w, 3) for w in unique_words] 
print(list(zip(unique_words[:20], shift_cipher[:20])))  # print sample

gross


NameError: name 'shift_letters' is not defined

In [14]:
import torch
from collections import Counter

# Sample text data
text = "This is an example sentence. This sentence is an example."

# 1. Create a Vocabulary
words = text.lower().split()
word_counts = Counter(words)
vocabulary = {word: i + 2 for i, (word, _) in enumerate(word_counts.items())}
vocabulary['<UNK>'] = 0
vocabulary['<PAD>'] = 1

# 2. Tokenize the Text (already done in this example)

# 3. Numericalize the Tokens
numericalized_text = [vocabulary[word] if word in vocabulary else vocabulary['<UNK>'] for word in words]

# 4. Create a Tensor
text_tensor = torch.tensor(numericalized_text)

print("Vocabulary:", vocabulary)
print("Numericalized Text:", numericalized_text)
print("Tensor:", text_tensor)

Vocabulary: {'this': 2, 'is': 3, 'an': 4, 'example': 5, 'sentence.': 6, 'sentence': 7, 'example.': 8, '<UNK>': 0, '<PAD>': 1}
Numericalized Text: [2, 3, 4, 5, 6, 2, 7, 3, 4, 8]
Tensor: tensor([2, 3, 4, 5, 6, 2, 7, 3, 4, 8])


In [29]:
"{}, {}".format(2, 3)

'2, 3'